In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Fixação da semente analítica para consistência estatística
np.random.seed(42)

# Vinculação dos módulos internos da biblioteca src
sys.path.append(os.path.abspath('../'))
from src.features import calcular_força_historica, extrair_metricas_elenco
from src.simulation import executar_monte_carlo_2026

print("Dependências importadas e ambiente configurado com sucesso.")

In [ ]:
# Carregando seus arquivos reais
df_matches = pd.read_csv("../data/raw/all_matches.csv")
df_ranking = pd.read_csv("../data/raw/fifa_ranking_2024.csv")
df_players = pd.read_csv("../data/raw/male_players.csv")

# TRATAMENTO ANTES DE COLOCAR NO DATASET: Padronizar nomes dos países
mapa_nomes_ranking = {
    'USA': 'United States',
    'South Korea': 'Korea Republic',
    'Ivory Coast': "Côte d'Ivoire"
}
df_ranking['country_full'] = df_ranking['country_full'].replace(mapa_nomes_ranking)

mapa_nomes_players = {
    'Korea Republic': 'South Korea',
    "Côte d'Ivoire": 'Ivory Coast'
}
df_players['Nation'] = df_players['Nation'].replace(mapa_nomes_players)

# Agora sim o ambiente está 100% limpo para gerar os arquivos finais!
print("Dados tratados e prontos para processamento.")

In [ ]:
config_copa_2026 = {
    "name": "2026 FIFA World Cup",
    "format": "48_team",
    "groups": {
        "A": ["Mexico", "South Africa", "South Korea", "Wales"],
        "B": ["Canada", "Ukraine", "Qatar", "Switzerland"],
        "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
        "D": ["United States", "Paraguay", "Australia", "Greece"],
        "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
        "F": ["Netherlands", "Japan", "Turkey", "Tunisia"],
        "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
        "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
        "I": ["France", "Senegal", "Norway", "Bolivia"],
        "J": ["Argentina", "Algeria", "Austria", "Jordan"],
        "K": ["Portugal", "Jamaica", "Uzbekistan", "Colombia"],
        "L": ["England", "Croatia", "Ghana", "Panama"]
    }
}

# Planificação das equipes elegíveis para validação de dados
todas_selecoes = [time for grupo in config_copa_2026["groups"].values() for time in grupo]
print(f"Configuração carregada. Total de seleções mapeadas: {len(todas_selecoes)}")

In [ ]:
PATH_RAW = "../data/raw/"

df_matches = pd.read_csv(f"{PATH_RAW}matches.csv")
df_players = pd.read_csv(f"{PATH_RAW}players.csv")

# Uniformização ortográfica obrigatória de strings de países
mapa_nomes = {
    'United States': 'United States', 'South Korea': 'South Korea',
    'Ivory Coast': 'Ivory Coast', 'Curaçao': 'Curaçao'
}
df_matches['home_team'] = df_matches['home_team'].replace(mapa_nomes)
df_matches['away_team'] = df_matches['away_team'].replace(mapa_nomes)
df_players['Nation'] = df_players['Nation'].replace(mapa_nomes)

print("Camada de dados brutos carregada e strings de texto normalizadas.")

In [ ]:
print("Calculando ratings de força Elo históricos...")
dicionario_elo = calcular_força_historica(df_matches, n_anos=10)

dados_processados = []

for time in todas_selecoes:
    ovr_medio, quantidade_craques = extrair_metricas_elenco(df_players, time)
    elo_calculado = dicionario_elo.get(time, 1500.0) # Fallback padrão se não localizado
    
    dados_processados.append({
        'selecao': time,
        'elo': elo_calculado,
        'overall_titulares': ovr_medio,
        'jogadores_elite': quantidade_craques
    })

df_dataset_final = pd.DataFrame(dados_processados).set_index('selecao')

# Salvando a matriz final consolidada
os.makedirs("../data/processed/", exist_ok=True)
df_dataset_final.to_csv("../data/processed/dataset_final.csv")

print("Matriz final unificada de atributos gerada.")
display(df_dataset_final.head())

In [ ]:
print("Iniciando processamento estocástico de Monte Carlo (10.000 execuções)...")
df_probabilidades_finais = executar_monte_carlo_2026(
    df_dataset_final, 
    config_copa_2026["groups"], 
    n_simulacoes=50
)

print("Simulações estabilizadas.")
display(df_probabilidades_finais.sort_values(by='Campeão', ascending=False))

In [ ]:
# Filtra as 12 principais seleções ordenadas pela maior probabilidade de título
df_grafico = df_probabilidades_finais.sort_values(by='Campeão', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(16, 8), dpi=100)
indices = np.arange(len(df_grafico.index))
largura = 0.20

barras_o = ax.bar(indices - 1.5*largura, df_grafico['Oitavas'], largura, label='Oitavas (%)', color='#cbd5e1')
barras_q = ax.bar(indices - 0.5*largura, df_grafico['Quartas'], largura, label='Quartas (%)', color='#94a3b8')
barras_s = ax.bar(indices + 0.5*largura, df_grafico['Semi'], largura, label='Semifinal (%)', color='#3b82f6')
barras_c = ax.bar(indices + 1.5*largura, df_grafico['Campeão'], largura, label='Campeão (%)', color='#1d4ed8')

ax.set_ylabel('Probabilidade Estatística de Avanço (%)', fontsize=12, fontweight='bold')
ax.set_title('Previsão Estocástica da Copa do Mundo 2026 - Distribuição de Probabilidades por Fase', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(indices)
ax.set_xticklabels(df_grafico.index, fontsize=11, rotation=30, ha='right', fontweight='bold')
ax.set_ylim(0, 105)
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.3)

def aplicar_valores(retangulos):
    for r in retangulos:
        h = r.get_height()
        if h > 1.0: # Exibe rótulo apenas se a chance for maior que 1% para manter clareza visual
            ax.annotate(f'{h:.1f}%',
                        xy=(r.get_x() + r.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=8, rotation=90)

aplicar_valores(barras_o)
aplicar_valores(barras_q)
aplicar_valores(barras_s)
aplicar_valores(barras_c)

plt.tight_layout()
plt.show()